# Simu-Learn - Autonomous Multi-Agent Learning System

## Welcome to the autonomous agentic simulation for a virtual learner

```
Main: Orchestrator → CurriculumDesigner → (for each topic: Tutor → QuizWorkflow)
QuizWorkflow: QuizMaster → Learner → Reviewer (repeated N times)
```

In [ ]:
%pip install -qU ipywidgets

In [1]:
import logging
import os
import sys
import random
from enum import StrEnum

from dotenv import load_dotenv
from openai import OpenAI


load_dotenv(override=True)

True

### Log Setup

In [2]:
logging.basicConfig(level=logging.WARNING)

valid_levels = ['DEBUG', 'INFO', 'WARNING', 'ERROR']
log_level = os.environ.get('LOG_LEVEL', 'DEBUG').upper()
log_level = log_level.upper() if log_level in valid_levels else 'DEBUG'

logger = logging.getLogger('simu-learn')
logger.setLevel(log_level)

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

## Utilities

### Secrets

In [3]:
def _get_secret_from_environment(name: str) -> str:
  secret = os.environ.get(name, '').strip()

  if secret:
      print(f'✅ Environment Variable: Found {name}')
      return secret

  print(f'❌ Environment Variable : {name} is not set')

  try:
    from google.colab import userdata

    secret = userdata.get(name)
    print(f'✅ Google Colab: Found {name}')
  except Exception as e:
    print(f'❌ Google Colab: {e}')

  return secret.strip()


def _get_secret_from_user(name: str) -> str:
  from getpass import getpass

  return getpass(f'Enter the secret value for {name}: ')

def get_secret(name: str) -> str:
  secret = _get_secret_from_environment(name)
  if not secret:
    secret = _get_secret_from_user(name)

  return secret

### Setup LLM Provider (Ollama)

🔑 Go to https://ollama.com/ to register for a **FREE** api key, if you need one

In [ ]:
class Provider(StrEnum):
    OLLAMA = 'Ollama'
    OPENROUTER = 'OpenRouter'
    CEREBRAS = 'Cerebras'

available_providers = [item.value for item in Provider]
provider_config: dict[Provider, tuple[str, str]] = {
    Provider.OLLAMA: ('OLLAMA_API_KEY', 'https://ollama.com/v1'),
    Provider.OPENROUTER: ('OPENROUTER_API_KEY', 'https://openrouter.ai/api/v1'),
    Provider.CEREBRAS: ('CEREBRAS_API_KEY', 'https://api.cerebras.ai/v1')
}

clients: dict[Provider, OpenAI] = {}

models: dict[Provider, list[str]] = {
    Provider.OLLAMA: [],
    Provider.OPENROUTER: [],
    Provider.CEREBRAS: [],
}

selection_state: dict[Provider, str | None] = {
    Provider.OLLAMA: 'gpt-oss:20b',
    Provider.OPENROUTER: 'openai/gpt-oss-20b:free',
    Provider.CEREBRAS: 'gpt-oss-120b',
}

DEFAULT_PROVIDER = Provider.CEREBRAS

selected_provider, selected_model, client = '', '', None


def get_desired_value_or_first_item(desire, options) -> str | None:
    logger.debug(f'Pick {desire} from {options}')
    selected = desire if desire in options else None
    if selected:
        return selected

    return options[0] if options else None


try:
    selected_provider = get_desired_value_or_first_item(DEFAULT_PROVIDER, available_providers)
    client = clients.get(selected_provider)
except Exception:
    logger.warning(f'❌ no provider configured and everything else from here will FAIL 🤦, I know you know this already.')


def load_models_if_needed(selected_provider):
    global selected_model, models, provider_config, clients

    provider = Provider(selected_provider)
    client = clients.get(provider, None)

    if not client:
        key_name, base_url = provider_config.get(provider)
        api_key = get_secret(key_name)

        if not api_key:
            return
        
        client = OpenAI(api_key=api_key, base_url=base_url)
        clients[provider] = client


    if client and not models.get(selected_provider):
        logging.info(f'📡 Fetching {selected_provider} models...')
        
        models[selected_provider] = [model.id for model in client.models.list()]
        selected_model = get_desired_value_or_first_item(
            selection_state[selected_provider], 
            models[selected_provider],
        )


load_models_if_needed(selected_provider)
client = clients.get(selected_provider, None)

logger.info(f'ℹ️ Provider: {selected_provider} Model: {selected_model}, Client: {client}')

2025-10-31 19:47:46,235 - simu-learn - DEBUG - Pick Cerebras from ['Ollama', 'OpenRouter', 'Cerebras']


DEBUG:simu-learn:Pick Cerebras from ['Ollama', 'OpenRouter', 'Cerebras']


Pick Cerebras from ['Ollama', 'OpenRouter', 'Cerebras']
Cerebras
2343 None
✅ Environment Variable: Found CEREBRAS_API_KEY
XXXX csk-wmvmnx5vdhy8dxyw36ermfxyhr3659dnk6dhrjp2fyemxt69 <openai.OpenAI object at 0x10c38bbf0> {<Provider.CEREBRAS: 'Cerebras'>: <openai.OpenAI object at 0x10c38bbf0>}
client <openai.OpenAI object at 0x10c38bbf0> []
YYYY
2025-10-31 19:47:49,302 - simu-learn - DEBUG - Pick gpt-oss-120b from ['qwen-3-235b-a22b-thinking-2507', 'qwen-3-coder-480b', 'gpt-oss-120b', 'qwen-3-235b-a22b-instruct-2507', 'llama3.1-8b', 'llama-3.3-70b', 'qwen-3-32b', 'llama-4-scout-17b-16e-instruct']


DEBUG:simu-learn:Pick gpt-oss-120b from ['qwen-3-235b-a22b-thinking-2507', 'qwen-3-coder-480b', 'gpt-oss-120b', 'qwen-3-235b-a22b-instruct-2507', 'llama3.1-8b', 'llama-3.3-70b', 'qwen-3-32b', 'llama-4-scout-17b-16e-instruct']


Pick gpt-oss-120b from ['qwen-3-235b-a22b-thinking-2507', 'qwen-3-coder-480b', 'gpt-oss-120b', 'qwen-3-235b-a22b-instruct-2507', 'llama3.1-8b', 'llama-3.3-70b', 'qwen-3-32b', 'llama-4-scout-17b-16e-instruct']
2025-10-31 19:47:49,318 - simu-learn - INFO - ℹ️ Provider: Cerebras Model: gpt-oss-120b, Client: <openai.OpenAI object at 0x10c38bbf0>


INFO:simu-learn:ℹ️ Provider: Cerebras Model: gpt-oss-120b, Client: <openai.OpenAI object at 0x10c38bbf0>


In [ ]:
import ipywidgets as widgets
from IPython.display import display

provider_selector = widgets.Dropdown(
    options=available_providers,
    value=get_desired_value_or_first_item(selected_provider, available_providers),
)

model_selector = widgets.Dropdown(
    options=models.get(selected_provider, []),
    value=get_desired_value_or_first_item(selection_state[selected_provider], models[selected_provider]),
)

def on_provider_change(change):
    global selected_provider, client, models

    logger.info(f'Provider changed to {change.new}')
    selected_provider = change.new
    load_models_if_needed(selected_provider)

    model_selector.options = models.get(selected_provider, [])
    model_selector.value = selection_state[selected_provider]


def on_model_change(change):
    global selected_provider, selected_model, selection_state

    selected_model = change.new
    selection_state[selected_provider] = selected_model
    logger.info(f'👉 Selected model: {selected_model}')


provider_selector.observe(on_provider_change, names='value')
model_selector.observe(on_model_change, names='value')

model_component = widgets.HBox(children=[provider_selector, model_selector])

display(model_component)

2025-10-31 19:48:07,555 - simu-learn - DEBUG - Pick Cerebras from ['Ollama', 'OpenRouter', 'Cerebras']


DEBUG:simu-learn:Pick Cerebras from ['Ollama', 'OpenRouter', 'Cerebras']


Pick Cerebras from ['Ollama', 'OpenRouter', 'Cerebras']
2025-10-31 19:48:07,603 - simu-learn - DEBUG - Pick gpt-oss-120b from ['qwen-3-235b-a22b-thinking-2507', 'qwen-3-coder-480b', 'gpt-oss-120b', 'qwen-3-235b-a22b-instruct-2507', 'llama3.1-8b', 'llama-3.3-70b', 'qwen-3-32b', 'llama-4-scout-17b-16e-instruct']


DEBUG:simu-learn:Pick gpt-oss-120b from ['qwen-3-235b-a22b-thinking-2507', 'qwen-3-coder-480b', 'gpt-oss-120b', 'qwen-3-235b-a22b-instruct-2507', 'llama3.1-8b', 'llama-3.3-70b', 'qwen-3-32b', 'llama-4-scout-17b-16e-instruct']


Pick gpt-oss-120b from ['qwen-3-235b-a22b-thinking-2507', 'qwen-3-coder-480b', 'gpt-oss-120b', 'qwen-3-235b-a22b-instruct-2507', 'llama3.1-8b', 'llama-3.3-70b', 'qwen-3-32b', 'llama-4-scout-17b-16e-instruct']


## Agents

In [6]:
import time
from typing import Any
from dataclasses import dataclass, field
from datetime import datetime, timedelta


@dataclass
class State:
    educational_level: str
    subject: str
    topics: list[str]
    topic: str
    mastery: dict[str, float]
    session_start: datetime
    history: list[Any]
    quiz: dict[str, str]
    learner_answer: str = field(default='')
    num_questions_per_topic: int = field(default=2)
    run_duration_minutes: int = field(default=5)


@dataclass
class AgentResponse:
    agent: str
    message: str
    current_state: State
    metadata: dict = field(default_factory=dict)


def add_message(role, text):
    timestamp = datetime.now().strftime('%H:%M:%S')
    print(f'\n[{timestamp}] {role}: {text}\n')
    time.sleep(1)

### Agents Definitions

In [ ]:

from abc import ABC, abstractmethod
from typing import override
import json
import copy

model = selected_model

class Agent(ABC):
    @abstractmethod
    def run(self, state: State) -> AgentResponse:
        ...

    def copy_state(self, state: State) -> State:
        return copy.deepcopy(state)


class CurriculumDesigner(Agent):

    @override
    def run(self, state: State) -> AgentResponse:
        prompt = f'''
        You are an experienced educationist in {state.subject}, tasked
        to generate 5 topics for a learner at the {state.educational_level} level as a JSON output.

        strictly produce **ONLY JSON output** with the list of topics:
        Example:
        ["topic 1", "topic 2"]
        '''

        resp = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            response_format={"type": "json_object"},
        )
        msg = resp.choices[0].message.content
        topics = json.loads(msg)

        current_state = self.copy_state(state)
        current_state.topics = topics

        return AgentResponse(
            agent='📚 CurriculumDesigner',
            message=f'{len(topics)} topics. {str(topics)}',
            current_state=current_state,
        )


class Tutor(Agent):

    @override
    def run(self, state: State) -> AgentResponse:
        prompt = f'''
        You are a tutor in {state.subject}.
        Explain {state.topic} a learner at the {state.educational_level} level.
        in not more than 2 short paragraphs
        '''

        resp = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
        )
        msg = resp.choices[0].message.content

        current_state = self.copy_state(state)

        return AgentResponse(
            agent='📘 Tutor',
            message=msg,
            current_state=current_state,
        )


class QuizMaster(Agent):

    @override
    def run(self, state: State) -> AgentResponse:
        prompt = f'''
        You are a tutor in {state.subject} for a learner at the {state.educational_level} level.

        Give 1 question on {state.topic} and it's corresponding answer in JSON output.

        Output a JSON object with the keys question and answer
        '''

        resp = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            response_format={"type": "json_object"},
        )
        msg = resp.choices[0].message.content
        print('Quiz: ', msg)
        quiz = json.loads(msg)
        current_state = self.copy_state(state)
        current_state.quiz = quiz

        return AgentResponse(
            agent='🧠 QuizMaster',
            message=quiz.get('question'),
            current_state=current_state,
        )



class Learner(Agent):

    @override
    def run(self, state: State) -> AgentResponse:
        should_be_correct = random.random() < 0.5

        prompt = f'''
        You are a learner studying {state.subject} at the {state.educational_level} level.

        Give a only short {"correct" if should_be_correct else "incorrect" } answer to the question on {state.topic}:

        {state.quiz['question']}
        '''

        resp = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
        )
        msg = resp.choices[0].message.content

        current_state = self.copy_state(state)
        current_state.learner_answer = msg

        return AgentResponse(
            agent='🧑 Learner',
            message=msg,
            current_state=current_state,
        )


class Reviewer(Agent):
    @override
    def run(self, state: State) -> AgentResponse:
        prompt = f'''
        You are an independent reviewer in {state.subject} at the {state.educational_level} level.

        The correct answer is: {state.quiz['answer']}
        The learner's answer is: {state.learner_answer}
        Rate the learner's answer between 0 to 100 with 100 being a perfect answer:

        Return only the rating value.
        '''

        resp = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
        )
        rating = float(resp.choices[0].message.content)

        current_state = self.copy_state(state)

        return AgentResponse(
            agent='🔍 Reviewer',
            message=f'Learner scored {rating}% {'✅' if rating > 50 else '❌'}',
            current_state=current_state,
        )

### Workflow

In [9]:
class QuizWorkflow(Agent):
    def __init__(self):
        super().__init__()
        print('⚙️ Initializing QuizWorkflow')

        self.quiz_master = QuizMaster()
        self.learner = Learner()
        self.reviewer = Reviewer()


    @override
    def run(self, state: State) -> AgentResponse:
        current_state = self.copy_state(state)

        num_questions = state.num_questions_per_topic
        print(f'\n⚙️ Beginning the quiz workflow for {num_questions} questions\n')
        for i in range(num_questions):

            response = self.quiz_master.run(current_state)
            add_message(response.agent, response.message)
            current_state = response.current_state

            response = self.learner.run(current_state)
            add_message(response.agent, response.message)
            current_state = response.current_state

            response = self.reviewer.run(current_state)
            add_message(response.agent, response.message)
            current_state = response.current_state

        print(f'\n⚙️ Completed workflow')

        return AgentResponse(
            agent='🔍 QuizWorkflow',
            message=f'Done running workflow%',
            current_state=current_state,
        )


class Orchestrator(Agent):
    def __init__(self):
        super().__init__()
        print('⚙️ Initializing Orchestrator')

        self.curriculum_designer = CurriculumDesigner()
        self.tutor = Tutor()
        self.quiz_workflow = QuizWorkflow()


    @override
    def run(self, state: State) -> AgentResponse:
        end_time = datetime.now() + timedelta(minutes=state.run_duration_minutes)
        add_message('🤖 Orchestrator', f'Starting autonomous session on "{state.subject}" ({state.educational_level})')

        response = self.curriculum_designer.run(state=state)
        add_message(response.agent, response.message)

        state = response.current_state
        topics = state.topics

        for topic_id, topic in enumerate(topics, start=1):
            if datetime.now() >= end_time:
                add_message('⏰ Time', 's up! Ending session.')

            state.topic = topic
            print(f'📌 Topic {topic_id}/{len(topics)}: {topic} ---\n')

            response = self.tutor.run(state)
            add_message(response.agent, response.message)

            response = self.quiz_workflow.run(state)

            print('\n\n' + '*' * 80 + '\n\n')

        print('✅ Session complete! Final mastery:', state.mastery)


In [10]:
def initialize_state() -> State:
    state = State(
        educational_level='Lower Primary',
        subject='Chemistry',
        mastery={},
        topic='',
        topics=[],
        history=[],
        quiz={'question': '', 'answer': ''},
        session_start = datetime.now()
    )

    return state

## Run

### Bare-bones no UI as that is not the focus

In [12]:
print(client, selected_provider, selected_model)

orchestrator = Orchestrator()
state = initialize_state()
orchestrator.run(state)

<openai.OpenAI object at 0x10c38bbf0> Cerebras llama3.1-8b
⚙️ Initializing Orchestrator
⚙️ Initializing QuizWorkflow

[19:56:50] 🤖 Orchestrator: Starting autonomous session on "Chemistry" (Lower Primary)


[19:56:51] 📚 CurriculumDesigner: 5 topics. ['What is water?', 'Things that melt and freeze', 'Colors of objects', 'Simple kitchen mixing experiments', 'Solids, liquids, and gases']

📌 Topic 1/5: What is water? ---


[19:56:53] 📘 Tutor: Water is a clear, tasteless liquid that we drink, use to wash our hands, and see in rivers, lakes, and rain. It is made of tiny building blocks called **molecules**, and each water molecule is put together from two parts called **hydrogen** and one part called **oxygen** (so its little formula is H₂O).  

Even though water looks the same all the time, it can change its shape: when it gets very cold it turns into **ice**, and when it gets very hot it becomes **steam** (a gas). All living things—people, plants, and animals—need water to stay healthy and 

### Minimal UI

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def add_message(role: str, text: str):
    timestamp = datetime.now().strftime('%H:%M:%S')
    with output_area:
        print(f'\n[{timestamp}] {role}: {text}\n')
    time.sleep(1)


level = widgets.Dropdown(
    options=['Primary', 'Secondary', 'Undergraduate', 'Graduate'],
    description='Educational Level',
)

subject = widgets.Text(description='Subject', placeholder= 'Chemistry')
duration = widgets.IntSlider(
    value=3,
    min=2,
    max=30,
    step=1,
    description='Duration (min):',
    style={'description_width': 'initial'}
)

button = widgets.Button(description='Learn', icon='check')
ui = widgets.HBox(children=[subject, level, duration, button])

output_area = widgets.Output(
    layout={
        'width': '100%',
        'height': '600px',
        'overflow_y': 'auto',
        'border': '1px solid #ddd',
        'padding': '10px',
        'background': '#f9f9f9'
    }
)


def on_learn_click(b):
    session_level = level.value
    session_subject = subject.value
    session_duration = duration.value

    with output_area:
        if not session_subject:
            print('⚠️ Please enter a subject.')
            return

        print(f'🤖 Starting session on {session_subject} at {session_level} level')

        orchestrator = Orchestrator()
        state = initialize_state()
        state.educational_level = session_level
        state.subject = session_subject
        state.run_duration_minutes = session_duration

        orchestrator.run(state)


button.on_click(on_learn_click)

display(model_component)
display(ui)
display(output_area)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…